# Writing Quantum Code with BlueQubit AI

*A worked example: from a one-sentence prompt to working, correct QAOA code.*

*Availability: BlueQubit AI is a quantum-native coding assistant available on the BlueQubit Platform.*

This notebook is not really about portfolio optimization. It is about BlueQubit AI: what it is, how you use it, and why the code it writes actually runs and solves the problem you asked about instead of just looking right. The running example is **quantum portfolio optimization**: a 10-asset QAOA walkthrough small enough to finish in a hosted notebook in a few minutes, but it still touches every spot where a general-purpose LLM tends to get quantum code quietly wrong: the QUBO encoding, the QUBO-to-Ising mapping, qubit ordering, the backend choice, and the optimization loop.

Here is the workflow the notebook walks through:

1. Ask BlueQubit AI, in plain English, for what you want.
2. Read the design rationale it returns before the code: which algorithm, which backend, which parameters, and why.
3. Run the generated code in the same hosted notebook to check it.
4. Look at the results, and at the quiet correctness fixes BlueQubit AI made for you.

## 0. What makes BlueQubit AI different

A general-purpose assistant writes quantum code the way it writes anything else: by guessing what looks right. In quantum computing that fails quietly. A circuit can run start to finish, hand back a clean histogram, and still answer a different question than the one you asked, because the qubit order got flipped, the QUBO picked up a wrong sign, or the simulator truncated the state without telling you.

BlueQubit AI is trained on and wired into the BlueQubit SDK and execution environment. In practice that buys you three things:

- It writes real `bluequbit` API calls with the right arguments instead of inventing them.
- It picks the backend for you, matching circuit size to the right simulation device (CPU statevector, GPU, or a tensor-network MPS with a sensible bond dimension) rather than defaulting to whatever showed up most in training.
- It writes code that runs, and that solves the problem you actually described.

The rest of the notebook puts that to the test. Each code cell is the kind of output BlueQubit AI produces, and the text around it is the reasoning it gives you alongside the code.

## 1. Why quantum computing for portfolio optimization

**Quantum computing for portfolio optimization** matters because the search space grows exponentially: 20 assets is manageable, 100 breaks classical search. Most classical methods either approximate poorly or take far too long.

QAOA (Quantum Approximate Optimization Algorithm) is one of the more promising routes for **QAOA portfolio optimization**. The recipe is short:

1. Encode the portfolio constraints as a QUBO (Quadratic Unconstrained Binary Optimization) problem.
2. Run it through a parameterized quantum circuit.
3. Tune the circuit parameters until the measurements start clustering around good allocations.

Writing this correctly is the fiddly part. The QUBO encoding, little-endian qubit ordering, backend selection, and bond-dimension tuning all have failure modes that throw no error. You get a wrong answer with no warning, and you can lose hours debugging a circuit that ran fine but solved the wrong problem. That is the kind of work BlueQubit AI is meant to handle.

In [ ]:
# The BlueQubit hosted notebook ships these preinstalled. Locally:
#   pip install bluequbit qiskit scipy numpy
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit

import bluequbit

print("bluequbit", bluequbit.__version__)

## 2. The prompt

This is the whole interaction with BlueQubit AI. One sentence.

> Prompt sent to BlueQubit AI
>
> *Optimize a portfolio of 10 assets. Explain the choice of algorithm, backend, and other design choices. Show me the best allocation.*

What comes back is more than code. BlueQubit AI starts with the design reasoning, then writes the full implementation. The sections below follow that order: the rationale first, then the code it generated.

## 3. What BlueQubit AI generated

> BlueQubit AI, design rationale
>
> - Algorithm: QAOA. Picking assets under a budget is a combinatorial binary optimization, which is a natural QUBO, and QUBOs are what QAOA targets. A depth-`p=1` ansatz shows the structure and keeps the parameter search down to two angles.
> - Backend: CPU statevector (`device="cpu"`). Ten qubits is `2^10 = 1024` amplitudes, which is trivial to simulate exactly. There is no reason to reach for GPU or a tensor-network (MPS) backend with bond-dimension tuning at this size, and exact simulation gives the optimizer the cleanest signal to work with. Past roughly 30 qubits, BlueQubit AI would switch to `mps.cpu` or `mps.gpu` with a tuned bond dimension.
> - Encoding: budget as a quadratic penalty. Instead of a hard constraint, the "pick `B` assets" rule goes into the QUBO as a penalty term, which keeps the problem unconstrained the way QAOA needs.
> - Ordering: little-endian aware parsing. BlueQubit returns measurement bitstrings with the least significant qubit first. The generated parser handles this so that QUBO variable `i` maps to qubit `i`. This is the most common silent bug in hand-written QAOA code.

The next cells are that implementation, in the order BlueQubit AI wrote them.

### 3a. QUBO portfolio optimization setup

Each asset is a binary variable: `1` means hold it, `0` means skip it. The QUBO matrix `Q` folds three things into one objective, `cost(x) = xᵀ Q x`:

1. Return: reward high-return assets (a linear term on the diagonal).
2. Risk: penalize correlated pairs through the covariance matrix (the quadratic term).
3. Budget: a penalty `(Σx − B)²` that pushes toward holding exactly `B` assets.

That is the heart of **QUBO portfolio optimization**: finding the binary `x` that minimizes `xᵀ Q x`.

In [ ]:
rng = np.random.default_rng(42)

num_assets = 10
asset_names = [f"A{i:02d}" for i in range(num_assets)]

# Synthetic but realistic-looking market data.
expected_returns = rng.uniform(0.05, 0.15, num_assets)          # per-asset expected return
_factor = rng.standard_normal((num_assets, num_assets))
covariance = (_factor @ _factor.T) / num_assets * 0.01          # PSD covariance matrix

budget = 4            # target number of assets to hold
risk_aversion = 0.5   # how strongly to penalize risk vs. chase return
penalty = 2.0         # strength of the budget constraint

# cost(x) = xᵀ Q x  with x_i ∈ {0,1} and x_i² = x_i.
# Q = risk_aversion·covariance  −  diag(returns)  +  penalty·(Σx − B)²
Q = risk_aversion * covariance.copy()
Q[np.diag_indices(num_assets)] -= expected_returns
Q += penalty * np.ones((num_assets, num_assets))        # x_i² and cross terms from the penalty
Q[np.diag_indices(num_assets)] += penalty * (-2 * budget)  # the −2B·Σx linear part

def qubo_cost(x):
    """Energy of a binary allocation under the QUBO."""
    x = np.asarray(x, dtype=float)
    return float(x @ Q @ x)

print(f"QUBO built for {num_assets} assets, target budget = {budget} holdings.")
print(f"Q is {Q.shape}, symmetric: {np.allclose(Q, Q.T)}")

### 3b. Ising model optimization from QUBO

QAOA's cost layer is built from Pauli-`Z` rotations, so the QUBO has to be rewritten in spin variables `z_i ∈ {+1, −1}` using `x_i = (1 − z_i) / 2`. That substitution turns `cost(x)` into an Ising Hamiltonian — the usual setup for **Ising model optimization** — with single-qubit fields `h_i` (the `RZ` gates) and couplings `J_ij` (the `RZZ` gates). The coefficients are easy to fumble by a factor of 2, and BlueQubit AI gets them right.

In [ ]:
def qubo_to_ising(Q):
    """Map a symmetric QUBO matrix to Ising fields h_i and couplings J_ij."""
    n = Q.shape[0]
    h = np.zeros(n)
    J = {}
    for i in range(n):
        off_diag = sum(Q[i, j] for j in range(n) if j != i)
        h[i] = -0.5 * Q[i, i] - 0.5 * off_diag
    for i in range(n):
        for j in range(i + 1, n):
            J[(i, j)] = 0.5 * Q[i, j]   # symmetric Q ⇒ (Q_ij + Q_ji)/4 = Q_ij/2
    return h, J

h, J = qubo_to_ising(Q)
print(f"Ising model: {len(h)} fields, {len(J)} couplings.")

### 3c. QAOA portfolio optimization circuit

QAOA alternates two layers for `p` rounds:

- a cost unitary `exp(-i·γ·H_C)`, the `RZ` and `RZZ` gates from the Ising model, which writes the objective into the phases;
- a mixer unitary `exp(-i·β·ΣX)`, the `RX` gates that push exploration across allocations.

The classical optimizer tunes the angles `γ` and `β`. Here we use `p = 1`, so two angles, which is what BlueQubit AI recommended for this size.

In [ ]:
def build_qaoa_circuit(params, h, J, n, p):
    """Construct the depth-p QAOA circuit for the portfolio Ising model."""
    gammas, betas = params[:p], params[p:]
    qc = QuantumCircuit(n)
    qc.h(range(n))                      # uniform superposition over all allocations
    for layer in range(p):
        g = gammas[layer]
        # Cost unitary: single-qubit fields then pairwise couplings.
        for i in range(n):
            if h[i] != 0.0:
                qc.rz(2 * g * h[i], i)
        for (i, j), j_ij in J.items():
            if j_ij != 0.0:
                qc.rzz(2 * g * j_ij, i, j)
        # Mixer unitary.
        qc.rx(2 * betas[layer], range(n))
    qc.measure_all()
    return qc

P_LAYERS = 1
demo = build_qaoa_circuit(np.array([0.3, 0.7]), h, J, num_assets, P_LAYERS)
print(f"QAOA circuit: {demo.num_qubits} qubits, depth {demo.depth()}, {demo.size()} gates.")

### 3d. Connect to BlueQubit and pick the backend

On a BlueQubit hosted notebook, `bluequbit.init()` authenticates automatically. Locally, pass your token (`bluequbit.init("YOUR_TOKEN")`) or set the `BLUEQUBIT_API_TOKEN` environment variable.

The backend here is `device="cpu"`, exact CPU statevector. BlueQubit AI chose it because 10 qubits is tiny, so exact simulation is fast and gives the optimizer a noise-free objective. The same `run` call can target `mps.cpu` or `mps.gpu` (tensor networks) or real quantum hardware just by changing the `device` string, and BlueQubit AI scales that choice to the circuit.

In [ ]:
# On a hosted BlueQubit notebook this needs no arguments.
bq = bluequbit.init()

DEVICE = "cpu"   # exact statevector, the right call for 10 qubits
SHOTS = 4096
print(f"Connected. Running QAOA on device='{DEVICE}' with {SHOTS} shots/eval.")

### 3e. The optimization loop, with the little-endian fix

Each optimizer step builds the circuit for the current angles, runs it on BlueQubit, and scores the returned samples against the QUBO.

The one catch, and the bug a general-purpose LLM almost always ships, is bit ordering. BlueQubit returns counts with the least significant qubit first. If you read the bitstring left to right as qubits `0..n-1`, you quietly transpose your variables and optimize the wrong thing. BlueQubit AI added the reversal (`bitstring[::-1]`) so that QUBO variable `i` lines up with qubit `i`. Skip it and nothing errors out; you just get a confidently wrong answer.

In [ ]:
def bitstring_to_x(bitstring):
    """BlueQubit orders bits least-significant-qubit-first.

    Reverse so that x[i] is the value of qubit i (i.e. QUBO variable i).
    This is the silent-bug fix BlueQubit AI applies automatically.
    """
    return np.array([int(b) for b in bitstring[::-1]], dtype=float)

_eval_count = 0

def expected_cost(params):
    """Mean QUBO cost of the samples produced by the QAOA circuit."""
    global _eval_count
    qc = build_qaoa_circuit(params, h, J, num_assets, P_LAYERS)
    counts = bq.run(qc, device=DEVICE, shots=SHOTS).get_counts()

    total, shots = 0.0, 0
    for bitstring, c in counts.items():
        total += qubo_cost(bitstring_to_x(bitstring)) * c
        shots += c

    _eval_count += 1
    return total / shots

# Two angles for p=1; COBYLA is derivative-free and well suited to noisy QAOA costs.
init_params = rng.uniform(0, np.pi, 2 * P_LAYERS)
result = minimize(
    expected_cost, init_params, method="COBYLA", options={"maxiter": 50},
)
print(f"Optimization done in {_eval_count} circuit evaluations.")
print(f"Best angles: gamma={result.x[:P_LAYERS]}, beta={result.x[P_LAYERS:]}")
print(f"Best expected cost: {result.fun:.4f}")

### 3f. Results: the best allocation

With the angles tuned, we run the circuit once more, take the most-sampled bitstrings, and report the lowest-cost allocation with its expected return and risk.

In [ ]:
final_counts = bq.run(
    build_qaoa_circuit(result.x, h, J, num_assets, P_LAYERS),
    device=DEVICE, shots=SHOTS,
).get_counts()

# Rank sampled allocations by their QUBO cost.
ranked = sorted(
    ((qubo_cost(bitstring_to_x(b)), b, c) for b, c in final_counts.items()),
    key=lambda t: t[0],
)
best_cost, best_bits, best_shots = ranked[0]
best_x = bitstring_to_x(best_bits)
selected = [asset_names[i] for i in range(num_assets) if best_x[i] == 1]

port_return = float(expected_returns @ best_x)
port_risk = float(best_x @ covariance @ best_x)

print("Best allocation found by BlueQubit AI's QAOA")
print("-" * 48)
print(f"  Selected assets : {selected}")
print(f"  Holdings        : {int(best_x.sum())} (target budget = {budget})")
print(f"  Expected return : {port_return:.4f}")
print(f"  Portfolio risk  : {port_risk:.4f}")
print(f"  QUBO cost       : {best_cost:.4f}")

# Sanity check against brute force, which is still feasible at 10 assets.
from itertools import product
brute = min(
    (qubo_cost(np.array(x)) for x in product([0, 1], repeat=num_assets)),
)
print(f"\n  Brute-force optimum cost: {brute:.4f}")
print(f"  QAOA matched optimum    : {np.isclose(best_cost, brute)}")

## 4. Deploy and verify, without leaving the notebook

Everything above ran on BlueQubit's hosted environment through the same `bq.run(...)` calls. That is the other half of what BlueQubit AI gives you: as soon as it writes the code, you can run it right there to confirm it works, with no local setup and no copying into a separate environment.

The loop is short:

1. Prompt BlueQubit AI.
2. Read the rationale.
3. Run the generated cells on hosted compute.
4. Check the results. Here, the QAOA allocation matched the brute-force optimum.

For circuits too large to brute-force, the loop is the same; you just lean on BlueQubit AI's backend choice (for example, MPS with a tuned bond dimension) instead of an exhaustive check.

## 5. Three things worth taking away

1. BlueQubit AI writes correct code, not a near-miss. The QUBO encoding, the QUBO-to-Ising coefficients, the little-endian parsing, the right SDK calls: these are the parts a general-purpose LLM gets wrong without saying so. BlueQubit AI gets them right because it is trained on and integrated with the BlueQubit environment specifically.

2. The backend decision is made for you. "CPU statevector at 10 qubits, switch to MPS with a tuned bond dimension past about 30" is the kind of judgment that takes a while to learn, and BlueQubit AI builds it into the code it writes.

3. Writing and running are one step. Because BlueQubit AI lives inside the hosted notebook, you go from a one-sentence prompt to working, checked quantum code in minutes. That speed is what matters most while the hardware keeps improving.

## 6. Next steps

This example is deliberately small so the whole loop finishes in minutes. The same workflow scales up:

- Scale the problem. Ask BlueQubit AI for 30, 50, or 100 assets, and it will move you off exact statevector onto a tensor-network backend with an appropriate bond dimension, and adjust the encoding to match.
- Deepen the ansatz. Ask for more QAOA layers (`p > 1`) for a closer approximation, and BlueQubit AI handles the extra angles and the heavier optimization.
- Change the objective. Add constraints like sector limits or transaction costs in plain English, and let BlueQubit AI re-derive the QUBO.
- Target real hardware. Swap `device="cpu"` for a quantum device string to run the same circuit on hardware through the same SDK.

Try it yourself: open a hosted notebook at app.bluequbit.io and give BlueQubit AI the one-sentence prompt from Section 2.